# Sudoku Model Predictions

Load the tuned one-hot-input MLP checkpoint used by `train.py` and compare a held-out test prediction with its solution.


In [ ]:
from pathlib import Path

import torch
from datasets import Features, Value, load_dataset

from model import NeuralNetwork
from utils import plot_sudoku

In [2]:
model_path = Path("sudoku_model.pt")
checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)

if "state_dict" in checkpoint:
    layer_sizes = checkpoint["layer_sizes"]
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint
    layer_sizes = [
        value.shape[0]
        for name, value in state_dict.items()
        if name.endswith(".weight")
    ][:-1]

net = NeuralNetwork(layer_sizes)
net.load_state_dict(state_dict)
net.eval()
print(f"Loaded {model_path} with hidden layers {layer_sizes}")

Loaded sudoku_model.pt with hidden layers [2048, 2048, 2048, 2048]


In [3]:
features = Features({"puzzle": Value("string"), "solution": Value("string")})
ds = load_dataset("certen/Sudoku-1M-Puzzles", features=features)

# Match the untouched test split used by the tuning search.
n_training = int(0.9 * len(ds["train"]))
n_held_out = len(ds["train"]) - n_training
order = torch.randperm(n_held_out, generator=torch.Generator().manual_seed(2026))
test_indices = order[n_held_out // 2:]
example_index = 0
example = ds["train"][n_training + test_indices[example_index].item()]
puzzle = torch.tensor([int(c) for c in example["puzzle"]], dtype=torch.long)
ground_truth = torch.tensor([int(c) for c in example["solution"]], dtype=torch.long)

In [ ]:
import matplotlib.pyplot as plt

with torch.inference_mode():
    prediction = net(puzzle.unsqueeze(0)).argmax(dim=1) + 1

plot_sudoku(prediction, ground_truth, puzzle)
plt.show()